[![cloudevel](img/cloudevel.png)](https://cloudevel.com)

# *Pipeline* de integración continua de una *API*.

Hasta aquí se han construido *workflows* de demostración: un saludo, expresiones y la gestión de artefactos. En este cuaderno se da el paso a un *pipeline* de **integración continua** (*CI*, *Continuous Integration*): cada cambio en el repositorio instala dependencias, ejecuta un linter y pruebas automatizadas, y deja un artefacto reutilizable. Tras un *CI* verde en la rama ```main```, un segundo *job* publica la imagen; ese paso se estudia en el cuaderno 15.

## Objetivos.

* Localizar la interfaz de programación de aplicaciones (*API*) de ejemplo en el directorio ```api/``` de este repositorio.
* Leer el *workflow* ```.github/workflows/ci.yml``` y relacionar el *job* ```verificar``` con lo visto en los cuadernos 05, 09 y 11, y con el privilegio mínimo del cuaderno 13.
* Ejecutar el *pipeline* en un repositorio propio y comprobar que las pruebas y el artefacto aparecen en la interfaz de *GitHub Actions*.

## La aplicación de ejemplo.

El directorio ```api/``` contiene una *API* *HTTP* (*Hypertext Transfer Protocol*) escrita en *Python* con *Flask*. Expone una ruta de salud ```GET /health``` y un conjunto sencillo de rutas para gestionar tareas, en formato *JSON* (*JavaScript Object Notation*).

Sin ```DATABASE_URL``` (es el caso en el *Codespace*, en las pruebas habituales y en *Cloud Run*), el almacén vive en la memoria del proceso. Si esa variable está definida, las tareas se guardan en *PostgreSQL*. Las pruebas están en ```api/tests/``` y se ejecutan con *pytest*. La cobertura mínima configurada en ```api/pyproject.toml``` es del 80 %; el linter *ruff* incluye reglas de seguridad (```S```) y de errores comunes (```B```).

La respuesta de ```GET /health``` incluye ```"almacen"``` (```memoria``` o ```postgresql```), ```"status"``` y ```"version"```. Si hay base configurada y no responde, la ruta devuelve el código 503 con ```"status": "error"```.

Quien cursa el taller no necesita diseñar esa aplicación: la usa como objeto del *pipeline*. El archivo ```api/.env.example``` es solo una plantilla; el ```.env``` real no se versiona. Si falta la clave de *Flask*, la aplicación genera una aleatoria: no hay una clave conocida en el código.

La siguiente celda instalará las dependencias de desarrollo, copiará la plantilla de entorno y ejecutará el linter y las pruebas, desde el directorio ```api/```.

In [ ]:
cd api
python -m venv .venv
source .venv/bin/activate
pip install -r requirements-dev.txt
cp .env.example .env
ruff check . && ruff format --check .
pytest

Con el almacén en memoria, *pytest* omite las pruebas que requieren *PostgreSQL*. La aplicación queda en ```http://127.0.0.1:8080```. El puerto de escucha se toma de la variable de entorno ```PORT``` (el valor de referencia está documentado en ```api/.env.example```). En el contenedor, el proceso de arranque es *gunicorn* sobre ```wsgi:app```, con un solo *worker*.

La siguiente celda arranca la *API* en segundo plano en el *Codespace*, desde el directorio ```api/```, para no bloquear el cuaderno. La bitácora del servidor de desarrollo de *Flask* se escribe en ```/tmp/cd441-flask.log``` y su número de proceso (*PID*, *process identifier*) se guarda en ```/tmp/cd441-flask.pid``` para detenerlo después. Las últimas líneas de la bitácora deben incluir ```Running on http://127.0.0.1:8080```.

In [ ]:
export PORT=8080
nohup flask --app wsgi:app run --port "${PORT}" > /tmp/cd441-flask.log 2>&1 &
echo $! > /tmp/cd441-flask.pid
sleep 2
tail -n 5 /tmp/cd441-flask.log

La siguiente celda consulta la ruta de salud ```GET /health``` de la *API* que acaba de arrancar. La respuesta esperada es un *JSON* con ```"status":"ok"``` y ```"almacen":"memoria"```.

In [ ]:
curl -s "http://127.0.0.1:${PORT}/health"

La siguiente celda detiene el servidor de desarrollo con el *PID* guardado y libera el puerto 8080, que el cuaderno 15 vuelve a usar con el contenedor. Si la *API* ya no estaba en ejecución, el mensaje lo indica.

In [ ]:
kill "$(cat /tmp/cd441-flask.pid)" 2>/dev/null && echo "API detenida." || echo "La API no estaba en ejecución."
rm -f /tmp/cd441-flask.pid

## Qué debe hacer el *pipeline* de integración continua.

El *pipeline* de *CI* de este curso, definido en ```.github/workflows/ci.yml```, tiene dos *jobs*:

* ```verificar```, que se dispara ante un *push*, un *pull request* o una ejecución manual: instala las dependencias de desarrollo, ejecuta *ruff* (incluido ```ruff format --check```) y *pytest*, publica el informe de cobertura ```api/coverage.xml``` como artefacto ```cobertura-pytest``` y construye la imagen (```docker build```) **sin** publicarla, para detectar un ```Dockerfile``` roto ya en la integración continua. Si el linter o las pruebas fallan, el *job* falla.
* ```publicar-imagen```, que solo corre tras un ```verificar``` verde en un *push* a ```main``` y publica esa imagen en *GitHub Container Registry* (*GHCR*). El cuaderno 15 lo detalla.

El archivo declara ```permissions: contents: read``` por defecto y ```concurrency``` para cancelar una ejecución anterior de la misma rama. Las *actions* van fijadas por *SHA* (*Secure Hash Algorithm*), como se explicó en el cuaderno 13. El despliegue se trata en los cuadernos 16 y 17. Aquí el objetivo es **confiar en el cambio** antes de entregarlo.

## El *workflow* ```ci.yml```.

El archivo vigente es ```.github/workflows/ci.yml```.

La siguiente celda mostrará el contenido de referencia de ese *workflow*.

```yaml
# Integración continua de la API (cuaderno 14) y publicación de la imagen (cuaderno 15).
#
# Buenas prácticas que muestra este archivo:
#  - Actions fijadas por SHA completo de commit (el comentario dice la versión).
#    Una etiqueta como @v7 puede moverse; un SHA no. Dependabot propone las
#    actualizaciones (.github/dependabot.yml).
#  - `permissions:` mínimos: lectura por defecto; solo el job que publica pide
#    `packages: write`, y solo se ejecuta en `main` (nunca en pull requests).
#  - `concurrency`: un push nuevo cancela la ejecución anterior de la misma rama.
#  - Etiqueta de imagen INMUTABLE: el SHA del commit, nunca `latest`.
name: CI

on:
  push:
  pull_request:
  workflow_dispatch:

permissions:
  contents: read

concurrency:
  group: ci-${{ github.ref }}
  cancel-in-progress: true

jobs:
  verificar:
    runs-on: ubuntu-latest
    timeout-minutes: 15
    defaults:
      run:
        working-directory: api
    steps:
      - name: Checkout
        uses: actions/checkout@3d3c42e5aac5ba805825da76410c181273ba90b1 # v7.0.1
        with:
          # El token no se queda guardado en .git/config para pasos posteriores.
          persist-credentials: false

      - name: Configurar Python
        uses: actions/setup-python@6d4fd22b20465ccc0f7354476588e8df7c9ca0e0 # v7.1.0
        with:
          python-version: "3.12"
          cache: pip
          cache-dependency-path: api/requirements-dev.txt

      - name: Instalar dependencias
        run: |
          python -m pip install --upgrade pip
          python -m pip install -r requirements-dev.txt

      - name: Linter (ruff)
        run: |
          ruff check --output-format=github .
          ruff format --check .

      - name: Pruebas (pytest)
        run: python -m pytest

      - name: Publicar informe de cobertura
        if: always()
        uses: actions/upload-artifact@cf430e030ddbb5b0abf93d22962f4752f3646cd9 # v7.0.2
        with:
          name: cobertura-pytest
          path: api/coverage.xml
          if-no-files-found: ignore
          retention-days: 7

      - name: Construir imagen (sin publicar)
        working-directory: .
        run: docker build -t cd441-api:ci .

  publicar-imagen:
    # Publica en GitHub Container Registry (GHCR) solo tras un CI verde en main.
    # El paquete se deja PÚBLICO (Package settings → Change visibility): así el
    # anfitrión Linux y Cloud Run (vía Artifact Registry) descargan sin credenciales.
    # Esa misma imagen es la que se despliega en todos los destinos: se construye
    # una vez y se promociona, sin reconstruir por entorno.
    needs: verificar
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    timeout-minutes: 20
    permissions:
      contents: read
      packages: write # publicar en GHCR con el GITHUB_TOKEN temporal (sin PAT)
    steps:
      - name: Checkout
        uses: actions/checkout@3d3c42e5aac5ba805825da76410c181273ba90b1 # v7.0.1
        with:
          persist-credentials: false

      - name: Nombre de la imagen en minúsculas
        id: imagen
        env:
          REPOSITORIO: ${{ github.repository }}
        run: echo "nombre=ghcr.io/${REPOSITORIO,,}/api" >> "$GITHUB_OUTPUT"

      - name: Configurar Buildx
        uses: docker/setup-buildx-action@f87e5991a6d7451dcb8d9637bfbc97413f497069 # v4.4.1

      - name: Iniciar sesión en GHCR
        uses: docker/login-action@8aa5831fb8d48d8f76594c7ddde61ae58019f550 # v4.8.0
        with:
          registry: ghcr.io
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}

      - name: Construir y publicar (etiqueta = SHA del commit)
        uses: docker/build-push-action@c3c9e263c25d99ce0380d002d59b67737d91b0dc # v7.4.0
        with:
          context: .
          push: true
          tags: ${{ steps.imagen.outputs.nombre }}:${{ github.sha }}
          labels: |
            org.opencontainers.image.source=${{ github.server_url }}/${{ github.repository }}
            org.opencontainers.image.revision=${{ github.sha }}
            org.opencontainers.image.vendor=Cloudevel
          cache-from: type=gha
          cache-to: type=gha,mode=max
```

Notas:

* El privilegio de escritura en paquetes (```packages: write```) aparece **solo** en el *job* ```publicar-imagen```, y ese *job* no corre en un *pull request*.
* ```working-directory: api``` evita repetir ```cd api``` en los pasos de *Python*. El paso de ```docker build``` vuelve a la raíz del repositorio, porque ahí está el ```Dockerfile```.
* La versión de *Python* es la 3.12, la misma familia que usa la imagen.
* El *checkout* usa ```persist-credentials: false```.
* El artefacto se llama ```cobertura-pytest``` y se publica con ```if: always()```.

## Actividad.

1. Asegurarse de que el repositorio de trabajo contiene el directorio ```api/```, el ```Dockerfile``` en la raíz y el archivo ```.github/workflows/ci.yml```.
2. Hacer un *push* o abrir un *pull request* para disparar el *workflow* ```CI```.
3. Confirmar que el *job* ```verificar``` termina en verde y que el artefacto ```cobertura-pytest``` aparece en el resumen de la ejecución.
4. Provocar un fallo a propósito (por ejemplo, alterar una prueba de forma temporal), volver a ejecutar y comprobar que el *workflow* falla. El informe de cobertura, si llegó a generarse, puede seguir apareciendo como artefacto gracias a ```if: always()```.
5. En un *push* a ```main```, localizar además el *job* ```publicar-imagen``` (se estudia en el cuaderno 15).

Si el *workflow* no aparece, revisar que el archivo esté en ```.github/workflows/``` y que la extensión sea ```.yml``` o ```.yaml```.

## Cierre.

A partir de este cuaderno, cada cambio de la *API* queda sometido a linter, pruebas automáticas y una construcción de imagen de comprobación. El cuaderno 15 estudia el ```Dockerfile``` y la publicación en *GHCR*; los cuadernos 16 y 17 promocionan y despliegan esa misma imagen, sin reconstruirla por entorno.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2023-2026.</p>